# NBA Scrapper using py_ball

In [1]:
%pip install nba_api

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade pandas


Note: you may need to restart the kernel to use updated packages.


In [3]:
from nba_api.stats.static import teams
import pandas as pd
from datetime import datetime, timedelta
from src.config import *
from src.utils import *
import time

#today formated as YYYY-MM-DD
today = datetime.today().strftime('%Y-%m-%d')
today_precise = datetime.today().strftime('%Y-%m-%d_%H-%M-%S')


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/conda/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/conda/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/conda/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.10/site-packages/traitlets/config/application.py", line 982, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.10/site-packages/ipykernel/

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/conda/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/conda/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/conda/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.10/site-packages/traitlets/config/application.py", line 982, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.10/site-packages/ipykernel/

AttributeError: _ARRAY_API not found

## 1. Extraction des équipes NBA
#A. Récupérer les équipes

In [4]:

# Récupère toutes les équipes NBA
nba_teams = teams.get_teams()
teams_df = pd.DataFrame(nba_teams)

# Garde les colonnes principales pour la lisibilité
main_cols = ['id', 'full_name', 'abbreviation', 'city', 'state', 'year_founded']
teams_df = teams_df[main_cols]

# Affiche un aperçu
display(teams_df)



# Sauvegarde en CSV
#teams_df.to_csv('nba_teams.csv', index=False)

save_dataframe_to_csv(teams_df, DATA_TEAMS_DIR, prefix='nba_teams_', suffix=today_precise)


,id,full_name,abbreviation,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Atlanta,Georgia,1949
1,1610612738,Boston Celtics,BOS,Boston,Massachusetts,1946
2,1610612739,Cleveland Cavaliers,CLE,Cleveland,Ohio,1970
3,1610612740,New Orleans Pelicans,NOP,New Orleans,Louisiana,2002
4,1610612741,Chicago Bulls,CHI,Chicago,Illinois,1966
5,1610612742,Dallas Mavericks,DAL,Dallas,Texas,1980
6,1610612743,Denver Nuggets,DEN,Denver,Colorado,1976
7,1610612744,Golden State Warriors,GSW,Golden State,California,1946
8,1610612745,Houston Rockets,HOU,Houston,Texas,1967
9,1610612746,Los Angeles Clippers,LAC,Los Angeles,California,1970


'data/raw/teams/nba_teams__2025-05-22_15-03-52.csv'

## 2. Extraction & stockage des matchs NBA (d’une journée)

In [4]:
# 
# import pandas as pd

# # Récupère tous les matchs NBA (tu peux filtrer par saison ou team après)
# gamefinder = leaguegamefinder.LeagueGameFinder(league_id_nullable='00', season_nullable='2023-24')
# games = gamefinder.get_data_frames()[0]

# # Affiche un aperçu
# display(games.head())

# Tu peux filtrer sur REGULAR_SEASON ou PLAYOFFS si tu veux
# games = games[games['SEASON_TYPE'] == 'Regular Season']

# Sauvegarde en CSV


In [5]:
from nba_api.stats.endpoints import leaguegamefinder


#from 2000 to today

seasons = [
    '2000-01',
    '2001-02',
    '2002-03',
    '2003-04',
    '2004-05',
    '2005-06',
    '2006-07',
    '2007-08',
    '2008-09',
    '2009-10',
    '2010-11',
    '2011-12',
    '2012-13',
    '2013-14',
    '2014-15',
    '2015-16',
    '2016-17',
    '2017-18',
    '2018-19',
    '2019-20',
    '2020-21', 
    '2021-22',
    '2022-23',
    '2023-24',
    '2024-25',
    '2025-26',
    '2026-27',
    ]
all_games = []
for season in seasons:
    print(f"Extraction saison {season}")
    gamefinder = leaguegamefinder.LeagueGameFinder(league_id_nullable='00', season_nullable=season)
    games = gamefinder.get_data_frames()[0]
    games['SEASON'] = season
    all_games.append(games)
    # petite pause pour être safe
    time.sleep(2)

df_games = pd.concat(all_games, ignore_index=True)

# Filter for only regular season and playoff NBA games
df_games = df_games[df_games['GAME_ID'].astype(str).str.startswith(('002','004', '005'))]

# Optional: sort by date
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])
df_games = df_games.sort_values(by='GAME_DATE', ascending=True)

save_dataframe_to_csv(df_games, DATA_GAMES_DIR, prefix='nba_games_', suffix=today_precise)

Extraction saison 2000-01
Extraction saison 2001-02
Extraction saison 2002-03
Extraction saison 2003-04
Extraction saison 2004-05
Extraction saison 2005-06
Extraction saison 2006-07
Extraction saison 2007-08
Extraction saison 2008-09
Extraction saison 2009-10
Extraction saison 2010-11
Extraction saison 2011-12
Extraction saison 2012-13
Extraction saison 2013-14
Extraction saison 2014-15
Extraction saison 2015-16
Extraction saison 2016-17
Extraction saison 2017-18
Extraction saison 2018-19
Extraction saison 2019-20
Extraction saison 2020-21
Extraction saison 2021-22
Extraction saison 2022-23
Extraction saison 2023-24
Extraction saison 2024-25
Extraction saison 2025-26
Extraction saison 2026-27


/tmp/ipykernel_36/66654322.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_games = pd.concat(all_games, ignore_index=True)


'data/raw/games/nba_games__2025-05-18_21-30-11.csv'

## 3 Extraction des stats joueurs par match (boxscore)

In [5]:
from nba_api.stats.endpoints import boxscoretraditionalv2
import os
import random

# Make sure output directories exist
os.makedirs(DATA_BOXSCORES_BATCHES_DIR, exist_ok=True)

os.makedirs(ERROR_LOG_FOLDER, exist_ok=True)

# Get latest games file
games_file = get_latest_file(DATA_GAMES_DIR)
print(f"Using games file: {games_file}")
games_df = pd.read_csv(games_file, dtype={'GAME_ID': str})

games_df['GAME_ID'] = games_df['GAME_ID'].str.zfill(10)

game_ids = games_df['GAME_ID'].unique().tolist()

# Resume logic: find already processed batches
processed_batches = [
    int(f.split('_')[-1].split('.')[0]) 
    for f in os.listdir(DATA_BOXSCORES_BATCHES_DIR) 
    if f.startswith('boxscores_players_batch_') and f.endswith('.csv')
]
start_batch = max(processed_batches) if processed_batches else 0
start_idx = start_batch * BATCH_SIZE

all_stats = []
batch_num = start_batch
error_log = []

for idx, gid in enumerate(game_ids[start_idx:], start=start_idx):
    try:
        
        #get current time
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        print(f"[{idx + 1}/{len(game_ids)}] Extracting GAME_ID: {gid}. Current time: {current_time}")
        box = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=gid, timeout=60)
        stats = box.player_stats.get_data_frame()
        stats['GAME_ID'] = gid
        all_stats.append(stats)
    except Exception as e:
        print(f"Error for GAME_ID {gid}: {e}")
        error_log.append((gid, str(e)))
        time.sleep(800)  # Longer pause on error
        continue

    # Anti-ban: sleep between each request
    time.sleep(random.uniform(3.7, 5.5))
    #time.sleep(0.2)  # Shorter pause between requests
    
    # Save a batch every BATCH_SIZE games or at the end
    if (idx + 1) % BATCH_SIZE == 0 or (idx + 1) == len(game_ids):
        batch_num += 1
        if all_stats:
            batch_df = pd.concat(all_stats, ignore_index=True)
            filename = f"boxscores_players_batch_{batch_num}"
            filepath = save_dataframe_to_csv(batch_df, DATA_BOXSCORES_BATCHES_DIR, prefix=filename)
            print(f"✅ Batch {batch_num} saved with {len(batch_df)} rows ({filepath})")
            all_stats = []  # Reset for next batch

        # Save error log for this batch
        if error_log:
            error_file = os.path.join(ERROR_LOG_FOLDER, f"errors_batch_{batch_num}.txt")
            with open(error_file, "a") as f:
                for err in error_log:
                    f.write(f"{err[0]}\t{err[1]}\n")
            error_log = []

        # Optional: long pause every 400 requests
        # if (idx + 1) % 400 == 0:
        #     print("Long pause to avoid rate limiting (2m)")
        #     time.sleep(120)


Using games file: data/raw/games/nba_games__2025-05-18_21-30-11.csv
[29526/32108] Extracting GAME_ID: 0022300100. Current time: 2025-05-22 15:04:29
[29527/32108] Extracting GAME_ID: 0022300096. Current time: 2025-05-22 15:04:33
[29528/32108] Extracting GAME_ID: 0022300099. Current time: 2025-05-22 15:04:38
[29529/32108] Extracting GAME_ID: 0022300101. Current time: 2025-05-22 15:04:42
[29530/32108] Extracting GAME_ID: 0022300111. Current time: 2025-05-22 15:04:46
[29531/32108] Extracting GAME_ID: 0022300110. Current time: 2025-05-22 15:04:51
[29532/32108] Extracting GAME_ID: 0022300102. Current time: 2025-05-22 15:04:56
[29533/32108] Extracting GAME_ID: 0022300103. Current time: 2025-05-22 15:05:01
[29534/32108] Extracting GAME_ID: 0022300105. Current time: 2025-05-22 15:05:05
[29535/32108] Extracting GAME_ID: 0022300104. Current time: 2025-05-22 15:05:10
[29536/32108] Extracting GAME_ID: 0022300107. Current time: 2025-05-22 15:05:15
[29537/32108] Extracting GAME_ID: 0022300108. Curren

/tmp/ipykernel_36/2837595841.py:57: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  batch_df = pd.concat(all_stats, ignore_index=True)


✅ Batch 1280 saved with 60 rows (data/raw/boxscores/batches/boxscores_players_batch_1280.csv)
[32026/32108] Extracting GAME_ID: 0022401193. Current time: 2025-05-22 22:01:56
[32027/32108] Extracting GAME_ID: 0022401188. Current time: 2025-05-22 22:02:00
[32028/32108] Extracting GAME_ID: 0022401199. Current time: 2025-05-22 22:02:05
[32029/32108] Extracting GAME_ID: 0022401192. Current time: 2025-05-22 22:02:10
[32030/32108] Extracting GAME_ID: 0022401195. Current time: 2025-05-22 22:02:14
[32031/32108] Extracting GAME_ID: 0022401187. Current time: 2025-05-22 22:02:20
[32032/32108] Extracting GAME_ID: 0022401191. Current time: 2025-05-22 22:02:24
[32033/32108] Extracting GAME_ID: 0022401198. Current time: 2025-05-22 22:02:30
[32034/32108] Extracting GAME_ID: 0022401200. Current time: 2025-05-22 22:02:34
[32035/32108] Extracting GAME_ID: 0022401190. Current time: 2025-05-22 22:02:39
[32036/32108] Extracting GAME_ID: 0022401196. Current time: 2025-05-22 22:02:44
[32037/32108] Extracting G

/tmp/ipykernel_36/2837595841.py:57: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  batch_df = pd.concat(all_stats, ignore_index=True)


✅ Batch 1281 saved with 227 rows (data/raw/boxscores/batches/boxscores_players_batch_1281.csv)
[32051/32108] Extracting GAME_ID: 0042400172. Current time: 2025-05-22 22:03:56
[32052/32108] Extracting GAME_ID: 0042400122. Current time: 2025-05-22 22:04:02
[32053/32108] Extracting GAME_ID: 0042400142. Current time: 2025-05-22 22:04:06
[32054/32108] Extracting GAME_ID: 0042400132. Current time: 2025-05-22 22:04:10
[32055/32108] Extracting GAME_ID: 0042400162. Current time: 2025-05-22 22:04:15
[32056/32108] Extracting GAME_ID: 0042400152. Current time: 2025-05-22 22:04:20
[32057/32108] Extracting GAME_ID: 0042400112. Current time: 2025-05-22 22:04:24
[32058/32108] Extracting GAME_ID: 0042400102. Current time: 2025-05-22 22:04:29
[32059/32108] Extracting GAME_ID: 0042400123. Current time: 2025-05-22 22:04:33
[32060/32108] Extracting GAME_ID: 0042400173. Current time: 2025-05-22 22:04:39
[32061/32108] Extracting GAME_ID: 0042400143. Current time: 2025-05-22 22:04:43
[32062/32108] Extracting 

## OPTIONNAL : Read erros on player stats to retry fetch and log erros again (run this until no error)

In [6]:
import os
import time
import random
import pandas as pd
from nba_api.stats.endpoints import boxscoretraditionalv2

# 1. Prepare folders
OLD_ERRORS_DIR = os.path.join(ERROR_LOG_FOLDER, f"old_errors_{today_precise}")
os.makedirs(OLD_ERRORS_DIR, exist_ok=True)

# 2. Collect all GAME_IDs from all error logs
error_files = [f for f in os.listdir(ERROR_LOG_FOLDER) if f.startswith('errors_batch_') and f.endswith('.txt')]
failed_game_ids = set()

for error_file in error_files:
    with open(os.path.join(ERROR_LOG_FOLDER, error_file), "r") as f:
        for line in f:
            gid = line.strip().split('\t')[0]
            if gid:
                failed_game_ids.add(gid)
    # Move the old log file to the archive folder
    os.rename(
        os.path.join(ERROR_LOG_FOLDER, error_file),
        os.path.join(OLD_ERRORS_DIR, error_file)
    )

print(f"Found {len(failed_game_ids)} failed GAME_IDs to retry.")

# 3. Retry extraction, with batch save and new error log
retry_batch_num = 1
all_stats = []
error_log = []

for idx, gid in enumerate(sorted(failed_game_ids)):
    try:
        print(f"[{idx+1}/{len(failed_game_ids)}] Retrying GAME_ID: {gid}")
        box = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=gid, timeout=30)
        stats = box.player_stats.get_data_frame()
        stats['GAME_ID'] = gid
        all_stats.append(stats)
    except Exception as e:
        print(f"Error for GAME_ID {gid}: {e}")
        error_log.append((gid, str(e)))
        time.sleep(8)  # Longer pause on error
        continue
    time.sleep(random.uniform(3, 5))  # Anti-ban pause

    # Batch save every BATCH_SIZE or at the end
    if (idx + 1) % BATCH_SIZE == 0 or (idx + 1) == len(failed_game_ids):
        if all_stats:
            batch_df = pd.concat(all_stats, ignore_index=True)
            filename = f"boxscores_players_retry_batch_{retry_batch_num}"
            filepath = os.path.join(DATA_BOXSCORES_BATCHES_DIR, f"{filename}.csv")
            batch_df.to_csv(filepath, index=False)
            print(f"✅ Retry batch {retry_batch_num} saved with {len(batch_df)} rows ({filepath})")
            all_stats = []
            retry_batch_num += 1

        # Log errors for this retry batch
        if error_log:
            error_file = os.path.join(ERROR_LOG_FOLDER, f"errors_retry_batch_{retry_batch_num}.txt")
            with open(error_file, "a") as f:
                for err in error_log:
                    f.write(f"{err[0]}\t{err[1]}\n")
            error_log = []

        # Optional: long pause every 100 requests
        if (idx + 1) % 100 == 0:
            print("Long pause to avoid rate limiting (5 minutes)")
            time.sleep(300)


Found 211 failed GAME_IDs to retry.
[1/211] Retrying GAME_ID: 0020100023
[2/211] Retrying GAME_ID: 0020100132
[3/211] Retrying GAME_ID: 0020100140
[4/211] Retrying GAME_ID: 0020100141
[5/211] Retrying GAME_ID: 0020100142
[6/211] Retrying GAME_ID: 0020100741
[7/211] Retrying GAME_ID: 0020100863
[8/211] Retrying GAME_ID: 0020100864
[9/211] Retrying GAME_ID: 0020100868
[10/211] Retrying GAME_ID: 0020100872
[11/211] Retrying GAME_ID: 0020200209
[12/211] Retrying GAME_ID: 0020200327
[13/211] Retrying GAME_ID: 0020200329
[14/211] Retrying GAME_ID: 0020200332
[15/211] Retrying GAME_ID: 0020200334
[16/211] Retrying GAME_ID: 0020200935
[17/211] Retrying GAME_ID: 0020201059
[18/211] Retrying GAME_ID: 0020201060
[19/211] Retrying GAME_ID: 0020201061
[20/211] Retrying GAME_ID: 0020201064
[21/211] Retrying GAME_ID: 0020300391
[22/211] Retrying GAME_ID: 0020300505
[23/211] Retrying GAME_ID: 0020300508
[24/211] Retrying GAME_ID: 0020300513
[25/211] Retrying GAME_ID: 0020300515
✅ Retry batch 1 saved w

## Merge boxscores batches in one file 

In [5]:
import os
import pandas as pd

# Liste tous les fichiers batchs dans ton dossier
all_boxscores_files = [
    os.path.join(DATA_BOXSCORES_BATCHES_DIR, f) for f in os.listdir(DATA_BOXSCORES_BATCHES_DIR) if f.endswith('.csv')
]
print(f"{len(all_boxscores_files)} batchs trouvés.")

# Concatène tout dans un seul DataFrame
dfs = [pd.read_csv(f, low_memory=False, dtype={'GAME_ID': str}) for f in all_boxscores_files]
boxscores_df = pd.concat(dfs, ignore_index=True)


save_dataframe_to_csv(boxscores_df, DATA_BOXSCORES_BATCHES_MERGED_DIR, prefix='boxscores_full_', suffix=today_precise)

print("✅ Merge & clean de tous les boxscores terminé !")



1293 batchs trouvés.
✅ Merge & clean de tous les boxscores terminé !
